In [19]:
from langchain_core.runnables import RunnableLambda, RunnableBranch, RunnableParallel, branch
from dataclasses import Field

from  langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
from openai import OpenAI
import httpx

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel
from typing import Literal
import uuid
import base64
import ssl



# This function will load all the variables from the .env file and will
# make them available in the os.environ dictionary (env variables)
load_dotenv()
ssl_cert_file=""
client= httpx.Client(
        verify=ssl.create_default_context(cafile=ssl_cert_file)
    )

client_id=os.environ.get('LLM_CLIENT_ID')
client_secret=os.environ.get('LLM_CLIENT_SECRET')



auth_string = f"{client_id}:{client_secret}"
auth_header = f"Basic {base64.b64encode(auth_string.encode()).decode()}"

llm_openai = ChatOpenAI()

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    ("human", "{input}")
])
str_parser = StrOutputParser()
# chain = prompt_template | llm_openai | str_parser
# result = chain.invoke({"input": "What is the capital of West Bengal?"})
# print(f"Result: {result}")

def dict_maker(text:str)->dict:
    return {'text':text}

run_lambda=RunnableLambda(dict_maker)
prompt_post = ChatPromptTemplate.from_messages([
    ("system", "You are a linkedin post generator."),
    ("human", "Create a post for the following text for LinkedIn:{text}")
])
#print(prompt_post)
chain=prompt_template|llm_openai|str_parser|run_lambda|prompt_post|llm_openai|str_parser
# result = chain.invoke("What is capital of france?")
# print(result)
chain_linkedin=prompt_post|llm_openai|str_parser

def insta_chain(text):
    # Handle both dict and llm_schema object
    if isinstance(text, dict):
        text_content = text.get('text', str(text))
    else:
        # It's an llm_schema object, convert to string
        text_content = str(text)
    
    insta_post = ChatPromptTemplate.from_messages([
        ("system", "You are a social media instagram post generator."),
        ("human", "Create a post for the following text for instagram :{text}")
        ])
    llm_openai = ChatOpenAI()
    str_parser = StrOutputParser()
    chain_insta=insta_post|llm_openai|str_parser
    result = chain_insta.invoke({"text": text_content})
    #print(f'instagram result:{result}')
    return result

insta_chain_lambda=RunnableLambda(insta_chain)
p_chain=(prompt_template|llm_openai|str_parser|run_lambda|RunnableParallel(branches={"linkedin":chain_linkedin,"insta":insta_chain_lambda}))
#resilt=p_chain.invoke("KGF")
def beautify(final_response:dict)-> dict:

    linkedin_response = final_response['branches']['linkedin']
    instagram_response = final_response['branches']['insta']

    return {"linkedin": linkedin_response, "instagram": instagram_response}
beautify_runnable = RunnableLambda(beautify)

final_chain=p_chain|beautify_runnable
#resilt=final_chain.invoke("KGF Indian Movie")
#print(resilt)

class llm_schema(BaseModel):
    movie_summary_flag: Literal["positive", "negative"]

llm_structured_output = llm_openai.with_structured_output(llm_schema)
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a movie review evaluator"),
    ("human", "Please categorize the movie review as positive or negative : {input}")])
llm_structured_output = llm_openai.with_structured_output(llm_schema)
def pydantic_json(input:llm_schema)-> str:
    return input.model_dump()['movie_summary_flag']

pydantic_json_lambda = RunnableLambda(pydantic_json)

conditional_chain=RunnableBranch((lambda x: 'positive' in str(x), chain_linkedin),insta_chain_lambda)
final_conditinal_chain=prompt_template|llm_structured_output|conditional_chain
result=final_conditinal_chain.invoke({"input":"I hate pushpa movie"})
print(result)

C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=llm_schema(movie_summary_flag='negative'), input_type=llm_schema])
  return self.__pydantic_serializer__.to_python(


**📽️ Movie Review: *[Title]* – Not Worth the Popcorn 🍿❌**

---

**Caption**  
I went in with high hopes, but this one fell flat. 😕 The story felt disjointed, the characters were one‑dimensional, and the pacing? A total snooze‑fest. If you’re looking for a night‑out that actually *entertains*, you might want to skip this one.  

**What didn’t work:**  
- **Plot:** More holes than Swiss cheese 🧀  
- **Acting:** Wooden performances that left me checking my watch ⏰  
- **Direction:** A confusing mess of scenes that never click together 🎬  

**Bottom line:** Save your tickets (and your time) for something that actually delivers. 🙅‍♀️

---

**Suggested Image**  
A still from the film with a bold red “X” overlay, or a split‑screen: left side a dramatic movie poster, right side a “thumbs‑down” emoji with a muted color palette.

---

**Hashtags**  
#MovieReview #FilmFail #SkipIt #CinemaSnooze #NotMyGenre #MovieNight #HonestOpinion #PopcornRegret #FilmCritique #NegativeVibes

---

**Call‑to‑Acti